In [22]:
import numpy as np
import pandas as pd
from pathlib import Path

## Combining the data

In [ ]:
# Importing the demand data with the correct data types
demandDf = pd.read_csv(
    "DataRaw/demand.csv",
    dtype={
        "Date": "float64",     # Excel serials may come as float
        "Period": "int16",
        "OperationalLessIndustrial": "float64",
        "Industrial": "float64",
    }
)


In [34]:
demandDf.head()

,Date,Period,OperationalLessIndustrial,Industrial
0,37257.0,1,3535.867064,1086.132936
1,37257.0,2,3383.499028,1088.500972
2,37257.0,3,3655.527552,1084.472448
3,37257.0,4,3510.446636,1085.553364
4,37257.0,5,3294.697156,1081.302844


In [ ]:
# Converting the excel serial date and time to a timestamp
midnight = pd.to_datetime(demandDf["Date"], unit="d", origin="1899-12-30")
demandDf["timestamp"] = midnight + pd.to_timedelta((demandDf["Period"] - 1) * 30, unit="m")
# Combining industrial and non industrial demand to get the overall demand
demandDf["Demand"] = demandDf["OperationalLessIndustrial"] + demandDf["Industrial"]

In [42]:
demandDf.tail()

,Date,Period,OperationalLessIndustrial,Industrial,timestamp,Demand
230779,42064.0,44,NaN,NaN,2015-03-01 21:30:00,NaN
230780,42064.0,45,NaN,NaN,2015-03-01 22:00:00,NaN
230781,42064.0,46,NaN,NaN,2015-03-01 22:30:00,NaN
230782,42064.0,47,NaN,NaN,2015-03-01 23:00:00,NaN
230783,42064.0,48,NaN,NaN,2015-03-01 23:30:00,NaN


In [ ]:
# Checking for data integrity with missing time periods and duplicate timestamps
per_day = demandDf.groupby(demandDf["timestamp"].dt.date).size()
assert per_day.ge(48).all(), "Some days have less than 48 half-hour periods"
dupes = demandDf["timestamp"].duplicated(keep=False).sum()
assert dupes == 0, f"Found {dupes} duplicated timestamps."

In [47]:
temperatureDf = pd.read_csv(
    "DataRaw/temperature.csv",
    dtype={
        "Date": "float64",     # Excel serials may come as float
        "Period": "int16",
        "Temp": "float64",
    }
)
temperatureDf.head()

,Date,Period,Temp
0,36526.0,1,13.7
1,36526.0,2,13.5
2,36526.0,3,13.3
3,36526.0,4,13.4
4,36526.0,5,13.5


In [48]:
# Converting the excel serial date and time to a timestamp
midnight = pd.to_datetime(temperatureDf["Date"], unit="d", origin="1899-12-30")
temperatureDf["timestamp"] = midnight + pd.to_timedelta((temperatureDf["Period"] - 1) * 30, unit="m")
temperatureDf.head()

,Date,Period,Temp,timestamp
0,36526.0,1,13.7,2000-01-01 00:00:00
1,36526.0,2,13.5,2000-01-01 00:30:00
2,36526.0,3,13.3,2000-01-01 01:00:00
3,36526.0,4,13.4,2000-01-01 01:30:00
4,36526.0,5,13.5,2000-01-01 02:00:00


In [61]:
holidayDf = pd.read_csv("DataRaw/holidays.txt", header=None,  names=["Date"])
# Converting the date column to datetime, and adding an indicator for holidays
holidayDf["Date"] = pd.to_datetime(holidayDf["Date"], dayfirst=True)
holidayDf["Holidays"] = True
holidayDf.head()

,Date,Holidays
0,2000-01-01,True
1,2000-01-26,True
2,2000-03-13,True
3,2000-04-21,True
4,2000-04-24,True


In [97]:
# Merging the demand and temperature data
df1 = pd.merge(demandDf, temperatureDf[["timestamp", "Temp"]], on="timestamp", how="left")
df1.head()

,Date,Period,OperationalLessIndustrial,Industrial,timestamp,Demand,Temp
0,37257.0,1,3535.867064,1086.132936,2002-01-01 00:00:00,4622.0,32.6
1,37257.0,2,3383.499028,1088.500972,2002-01-01 00:30:00,4472.0,32.6
2,37257.0,3,3655.527552,1084.472448,2002-01-01 01:00:00,4740.0,32.6
3,37257.0,4,3510.446636,1085.553364,2002-01-01 01:30:00,4596.0,32.6
4,37257.0,5,3294.697156,1081.302844,2002-01-01 02:00:00,4376.0,32.6


In [98]:
# Creating a common column and merging with holiday data
df1["date"] = pd.to_datetime(df1["timestamp"]).dt.date
holidayDf["date"] = pd.to_datetime(holidayDf["Date"]).dt.date
df = pd.merge(df1, holidayDf, on="date", how="left")
df["Holidays"] = df["Holidays"].fillna(False)
df.head()

/var/folders/jn/dz1dlbb55pv1crmy8zcmxqsw0000gn/T/ipykernel_38599/3161082180.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Holidays"] = df["Holidays"].fillna(False)


,Date_x,Period,OperationalLessIndustrial,Industrial,timestamp,Demand,Temp,date,Date_y,Holidays
0,37257.0,1,3535.867064,1086.132936,2002-01-01 00:00:00,4622.0,32.6,2002-01-01,2002-01-01,True
1,37257.0,2,3383.499028,1088.500972,2002-01-01 00:30:00,4472.0,32.6,2002-01-01,2002-01-01,True
2,37257.0,3,3655.527552,1084.472448,2002-01-01 01:00:00,4740.0,32.6,2002-01-01,2002-01-01,True
3,37257.0,4,3510.446636,1085.553364,2002-01-01 01:30:00,4596.0,32.6,2002-01-01,2002-01-01,True
4,37257.0,5,3294.697156,1081.302844,2002-01-01 02:00:00,4376.0,32.6,2002-01-01,2002-01-01,True


## Filtering the data only from January to June of 2014

In [ ]:
start_date = 41640.0
end_date = 41820.0
df["InRange"] = (df["Date_x"] >= start_date) & (df["Date_x"] <= end_date)
df = df.loc[df["InRange"]]
df.head()

,Date_x,Period,OperationalLessIndustrial,Industrial,timestamp,Demand,Temp,date,Date_y,Holidays,InRange
210384,41640.0,1,3914.647130,1009.352870,2014-01-01 00:00:00,4924.0,18.2,2014-01-01,2014-01-01,True,True
210385,41640.0,2,3672.549608,1004.450392,2014-01-01 00:30:00,4677.0,17.9,2014-01-01,2014-01-01,True,True
210386,41640.0,3,3497.539030,1004.460970,2014-01-01 01:00:00,4502.0,17.6,2014-01-01,2014-01-01,True,True
210387,41640.0,4,3339.144792,1003.855208,2014-01-01 01:30:00,4343.0,16.8,2014-01-01,2014-01-01,True,True
210388,41640.0,5,3204.312604,998.687396,2014-01-01 02:00:00,4203.0,16.3,2014-01-01,2014-01-01,True,True


In [104]:
df.drop(columns=["Date_x", "Date_y", "InRange"], inplace=True)
df.head()

,Period,OperationalLessIndustrial,Industrial,timestamp,Demand,Temp,date,Holidays,DOW
0,1,3914.647130,1009.352870,2014-01-01 00:00:00,4924.0,18.2,2014-01-01,True,Wednesday
1,2,3672.549608,1004.450392,2014-01-01 00:30:00,4677.0,17.9,2014-01-01,True,Wednesday
2,3,3497.539030,1004.460970,2014-01-01 01:00:00,4502.0,17.6,2014-01-01,True,Wednesday
3,4,3339.144792,1003.855208,2014-01-01 01:30:00,4343.0,16.8,2014-01-01,True,Wednesday
4,5,3204.312604,998.687396,2014-01-01 02:00:00,4203.0,16.3,2014-01-01,True,Wednesday


In [101]:
df["DOW"] = df["timestamp"].dt.day_name()

In [105]:
df.reset_index(drop=True, inplace=True)

In [106]:
df.tail()

,Period,OperationalLessIndustrial,Industrial,timestamp,Demand,Temp,date,Holidays,DOW
8683,44,5255.663988,1026.336012,2014-06-30 21:30:00,6282.0,10.2,2014-06-30,False,Monday
8684,45,5049.619246,1034.380754,2014-06-30 22:00:00,6084.0,10.0,2014-06-30,False,Monday
8685,46,4867.473156,1035.526844,2014-06-30 22:30:00,5903.0,10.1,2014-06-30,False,Monday
8686,47,5067.728750,1043.271250,2014-06-30 23:00:00,6111.0,10.1,2014-06-30,False,Monday
8687,48,5074.973196,1038.026804,2014-06-30 23:30:00,6113.0,10.0,2014-06-30,False,Monday
